In [ ]:
import shutil
from pathlib import Path

import torch

# The training kernel's output is mounted read-only under /kaggle/input.
candidates = sorted(Path("/kaggle/input").rglob("model_last.pt"))
assert candidates, f"no model_last.pt under {[p.name for p in Path('/kaggle/input').glob('*')]}"
source = candidates[0]
print("source     :", source)
print("size       : %.3f GB" % (source.stat().st_size / 1e9))

state = torch.load(str(source), map_location="cpu", weights_only=True)
print("keys       :", list(state.keys()))
print("update     :", state.get("update"))

ema = state["ema_model_state_dict"]
print("ema tensors:", len(ema))

# Keep initted/step: the EMA wrapper's own load_state_dict expects them, and
# they cost nothing. What matters is that no top-level "update" survives, so
# anything loading this treats it as weights rather than a resumable run.
out = Path("/kaggle/working/finetuned_ema.pt")
torch.save({"ema_model_state_dict": ema}, out)
del state, ema

check = torch.load(str(out), map_location="cpu", weights_only=True)
assert "update" not in check and "optimizer_state_dict" not in check
print("wrote      :", out, "%.3f GB" % (out.stat().st_size / 1e9))
print("tensors    :", len(check["ema_model_state_dict"]))
del check

# Nothing else should ride along and eat the output quota.
for stray in Path("/kaggle/working").iterdir():
    if stray != out:
        shutil.rmtree(stray) if stray.is_dir() else stray.unlink()
print("working    :", [p.name for p in Path("/kaggle/working").iterdir()])
